### Lesson 1: Simple ReAct Agent from Scratch

In [7]:
# !pip install dotenv
# !pip install cerebras-cloud-sdk

In [16]:
import os 
from dotenv import load_dotenv

load_dotenv()

cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
print("My API key is:", cerebras_api_key[:10])

My API key is: csk-9h5kk9


In [ ]:
from cerebras.cloud.sdk import Cerebras

def call_llm(messages, response_format=None, tools=None, model="qwen-3-235b-a22b-instruct-2507", temparature=0):
    client = Cerebras(api_key=cerebras_api_key)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temparature,
        response_format=response_format,
        tools=tools,
    )
    print(f"Token Usage: {response.usage}")
    return response.choices[0].message.content.strip()

In [19]:
messages=[{
        "role": "system",
        "content": "You are a helpful AI Assistant."
        },
        {
        "role": "user",
        "content": "Suggest some animes to binge watch."
        }]
response = call_llm(messages)
print(response)

Token Usage: ChatCompletionResponseUsage(completion_tokens=1143, prompt_tokens=29, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=1172)
Absolutely! Here’s a curated list of animes across different genres and styles that are perfect for binge-watching—each with strong storytelling, compelling characters, and high rewatch value:

### 🎯 **Action & Adventure**
1. **Attack on Titan (Shingeki no Kyojin)**  
   *Dark, intense, and full of twists. Follow humanity’s fight for survival against giant humanoid Titans.*  
   Why binge: Gripping plot, shocking reveals, and epic animation.

2. **Demon Slayer: Kimetsu no Yaiba**  
   *A boy becomes a demon slayer to save his sister. Stunning visuals and emotional depth.*  
   Why binge: Beautiful animation, fast-paced action, and heartfelt story.

3. **Jujutsu Kaisen**  
   *A boy swallows a cursed object and becomes a sorcerer in a world of supernatural threats.*  
   Why binge: Modern animation, g

In [23]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    def execute(self):
        return call_llm(self.messages)



In [20]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [22]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [24]:
abot = Agent(prompt)

In [25]:
result = abot("How much does a toy poodle weigh?")
print(result)

Token Usage: ChatCompletionResponseUsage(completion_tokens=32, prompt_tokens=233, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=265)
Thought: I should find the average weight of a toy poodle using the available action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [26]:
result = average_dog_weight("Toy Poodle")
print(result)

a toy poodles average weight is 7 lbs


In [27]:
next_prompt = "Observation: {}".format(result)

In [29]:
abot(next_prompt)

Token Usage: ChatCompletionResponseUsage(completion_tokens=15, prompt_tokens=287, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=302)


'Answer: A toy poodle weighs an average of 7 lbs.'

In [30]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [31]:
abot = Agent(prompt)

In [32]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

Token Usage: ChatCompletionResponseUsage(completion_tokens=57, prompt_tokens=247, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=304)


'Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together to get the combined weight. I will use the average_dog_weight action for each breed.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [33]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [34]:
abot(next_prompt)

Token Usage: ChatCompletionResponseUsage(completion_tokens=14, prompt_tokens=327, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=341)


'Action: average_dog_weight: Scottish Terrier\nPAUSE'

In [35]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [36]:
abot(next_prompt)

Token Usage: ChatCompletionResponseUsage(completion_tokens=40, prompt_tokens=361, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=401)


'Thought: I now have the average weights for both breeds. I will calculate their combined weight by adding the two values together.\nAction: calculate: 37 + 20\nPAUSE'

In [37]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [38]:
abot(next_prompt)

Token Usage: ChatCompletionResponseUsage(completion_tokens=23, prompt_tokens=416, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=439)


'Answer: The combined average weight of a Border Collie and a Scottish Terrier is 57 lbs.'

In [39]:
import re
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [40]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i+=1
        result = bot(next_prompt)
        print(result)
        actions=[
            action_re.match(a) for a in result.split("\n") if action_re.match(a)
        ]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return


In [41]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Token Usage: ChatCompletionResponseUsage(completion_tokens=57, prompt_tokens=247, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=304)
Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together to get the combined weight. I will use the average_dog_weight action for each breed.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Token Usage: ChatCompletionResponseUsage(completion_tokens=14, prompt_tokens=327, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=341)
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Token Usage: ChatCompletionResponseUsage(completion_tokens=40, prompt_tokens=361, prompt_tokens_details=ChatCompletionResponseUsagePromptTo